
# Tiny-GPT training from scratch on WikiText-2

In this assignment we will implement and train a small decoder-only Transformer language model from scratch on **WikiText-2** in **Google Colab**.


In [ ]:
!pip -q install datasets transformers


## 1. Imports and environment setup

This section:
- disables tokenizer parallelism warnings in Google Colab,
- imports required libraries,
- selects the runtime device.


**Note:** Ensure that runtime is cuda otherwise training will be painfully slow.


In [ ]:

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset
from transformers import AutoTokenizer
import torch, math, glob, random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if device != "cuda":
  print("Training will be slow since CUDA is not available....")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



## 2. Optional Google Drive mount for checkpoints

In Colab, training may be interrupted by runtime resets or disconnects.  
This section mounts Google Drive and prepares a checkpoint directory so training can resume later.

If Drive is unavailable, the notebook falls back to a local path.


In [ ]:

use_drive = True

if use_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base_dir = "/content/drive/MyDrive/gpt_wikitext2_solution"
    except Exception:
        base_dir = "/content/gpt_wikitext2_solution"
else:
    base_dir = "/content/gpt_wikitext2_solution"

os.makedirs(base_dir, exist_ok=True)
ckpt_dir = os.path.join(base_dir, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

print("base_dir:", base_dir)
print("checkpoint directory:", ckpt_dir)



## 3. Load WikiText-2 from Hugging Face

We use the raw WikiText-2 split:

- dataset: `Salesforce/wikitext`
- config: `wikitext-2-raw-v1`

The raw version contains plain text rather than the processed word-level benchmark form, which makes it appropriate for custom tokenizer-based language modeling.


In [ ]:

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
dataset


In [ ]:

for split in dataset:
    print(split, len(dataset[split]))
print()
print("sample train rows:")
print(dataset["train"][0])
print(dataset["train"][1])



## 4. Initialize the tokenizer

We use the GPT-2 tokenizer for convenience. If no pad token is defined, we reuse the EOS token as the pad token.


In [ ]:

tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("vocab_size:", tokenizer.vocab_size)
print("eos_token_id:", tokenizer.eos_token_id)
print("pad_token_id:", tokenizer.pad_token_id)



## 5. Preprocess the text corpus

The preprocessing pipeline is:

1. remove empty or whitespace-only strings,
2. tokenize each text example,
3. append an EOS token after each example,
4. concatenate all token IDs into one flat stream.

This flat-stream setup is simple and effective for next-token prediction.


In [ ]:
def clean(xs):
    return [x for x in xs if x and x.strip()]

train_text = clean(dataset["train"]["text"])
val_text = clean(dataset["validation"]["text"])

def encode(xs):
    ids = []
    for t in xs:
        tok = tokenizer.encode(t, add_special_tokens=False)
        tok.append(tokenizer.eos_token_id)
        ids.extend(tok)
    return ids

train_ids = encode(train_text)
val_ids = encode(val_text)

print("num train texts:", len(train_text))
print("num val texts:", len(val_text))
print()
print("num train tokens:", len(train_ids))
print("num val tokens:", len(val_ids))


## 6. Build the next-token-prediction dataset

Each example consists of:
- an input sequence `x` of length `block_size`,
- a target sequence `y` of length `block_size`.

The target is the input shifted by one token:
- `x = ids[i : i + block_size]`
- `y = ids[i + 1 : i + block_size + 1]`


In [ ]:
block_size = 128

class TokenDataset(Dataset):
    def __init__(self, ids, block):
        self.ids = ids
        self.block = block

    def __len__(self):
        return max(0, len(self.ids) - self.block)

    def __getitem__(self, i):
        x = ... # TODO
        y = ... # TODO
        return x, y

train_ds = ... # TODO
val_ds = ... # TODO

print("train examples:", len(train_ds))
print("val examples:", len(val_ds))


## 7. Create dataloaders

For Colab stability:
- `num_workers=4`
- tokenizer parallelism disabled


In [ ]:
batch_size = 64

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)

xb, yb = next(iter(train_loader))
print("train batch x shape:", xb.shape)
print("train batch y shape:", yb.shape)


## 8. Define model hyperparameters

These settings are intentionally small enough for Colab.


In [ ]:

vocab_size = tokenizer.vocab_size
n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.1
lr = 3e-4

assert n_embd % n_head == 0



## 9. Implement causal self-attention

This module performs:
- Q, K, V projection,
- multi-head reshape,
- scaled dot-product attention,
- causal masking,
- output projection.


In [ ]:
class Attention(nn.Module):
    def __init__(self, d, h, block, dropout=0.1):
        super().__init__()
        assert d % h == 0
        self.h = h
        self.head_dim = ... # TODO: compute head dimension

        # TODO: define the four linear projections
        self.q_proj = ...
        self.k_proj = ...
        self.v_proj = ...
        self.out_proj = ...
        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block, block))
        self.register_buffer("mask", mask.view(1, 1, block, block))

    def forward(self, x):
        B, T, C = x.shape

        # TODO: project inputs to q, k, v
        q = ...
        k = ...
        v = ...

        # TODO: reshape q, k, v into multi-head format
        q = ...
        k = ...
        v = ...

        # TODO: compute scaled dot-product attention scores
        att = ...

        # TODO: apply the causal mask
        att = ...

        # TODO: normalize attention scores and apply dropout
        att = ...
        att = ...

        # TODO: apply attention to the values
        y = ...

        # TODO: merge the heads back together
        y = ...

        # TODO: apply the final output projection
        y = ...
        return y


## 10. Implement the feed-forward network

A standard Transformer MLP:
- expand from `d` to `4d`,
- apply GELU,
- project back to `d`,
- apply dropout.


In [ ]:
class MLP(nn.Module):
    def __init__(self, d, dropout=0.1):
        super().__init__()
        # TODO: define the two linear layers and activation
        self.fc1 = ...
        self.fc2 = ...
        self.act = ...
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: implement the MLP forward pass
        return ...


## 11. Implement one Transformer block

We use a pre-norm residual block layout:
- LayerNorm
- attention
- residual addition
- LayerNorm
- MLP
- residual addition


In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, block, dropout=0.1):
        super().__init__()
        # TODO: define the layer norms, attention module, and MLP
        self.ln1 = ...
        self.attn = ...
        self.ln2 = ...
        self.mlp = ...

    def forward(self, x):
        # TODO: implement the pre-norm residual block
        x = ...
        x = ...
        return x

## 12. Implement the full GPT model

The model includes:
- token embeddings,
- positional embeddings, which encode the order of tokens in a sequence, allowing the model to understand the position of each word. These are **learnable** positional embeddings.
- stacked Transformer blocks,
- final LayerNorm,
- language-model head.

The forward pass returns:
- `logits`
- `loss` if targets are provided

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, block, d=256, h=4, L=4, dropout=0.1):
        super().__init__()
        self.block = block

        # TODO: define token embedding, positional embedding, dropout,
        # Transformer blocks, final layer norm, and LM head
        self.token_emb = ...
        self.pos_emb = ...
        self.drop = ...
        self.blocks = ...
        self.ln_f = ...
        self.lm_head = ...

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block:
            raise ValueError(f"sequence length {T} exceeds block size {self.block}")

        # TODO: create position indices
        pos = ...

        # TODO: combine token and positional embeddings and apply dropout
        x = ...

        # TODO: pass x through all Transformer blocks
        ...

        # TODO: compute logits using the final layer norm and LM head
        logits = ...

        loss = None
        if targets is not None:
            # TODO: compute next-token cross-entropy loss
            loss = ...
        return logits, loss


## 13. Instantiate model and optimizer


In [ ]:
# TODO: instantiate the GPT model and AdamW optimizer

model = ...
opt = ...

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable parameters:", f"{num_params:,}")


## 14. Checkpoint utilities

Each checkpoint stores:
- model weights,
- optimizer state,
- training step,
- loss step and history

This is enough to resume the training loop cleanly for this notebook.


In [ ]:
def get_checkpoint_path(step):
    return os.path.join(ckpt_dir, f"ckpt_step_{step:06d}.pt")

def save_checkpoint(model, opt, step, loss_steps, loss_history):
    state = {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "step": step,
        "loss_steps": loss_steps,
        "loss_history": loss_history,
    }
    path = get_checkpoint_path(step)
    torch.save(state, path)
    print("saved checkpoint:", path)

def latest_checkpoint():
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "ckpt_step_*.pt")))
    return ckpts[-1] if ckpts else None

def load_checkpoint(path, model, opt):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state["model"])
    opt.load_state_dict(state["optimizer"])
    start_step = state["step"]
    loss_steps = state.get("loss_steps", [])
    loss_history = state.get("loss_history", [])
    print("loaded checkpoint:", path, "| step:", start_step)
    return start_step, loss_steps, loss_history


## 15. Resume training if a checkpoint exists

If a checkpoint is found, training resumes from that step.
Otherwise, training starts from scratch.


In [ ]:
start_step = 0
loss_steps = []
loss_history = []

last_ckpt = latest_checkpoint()

if last_ckpt is not None:
    start_step, loss_steps, loss_history = load_checkpoint(last_ckpt, model, opt)
else:
    print("no checkpoint found, starting from scratch")


## 16. Training loop

This loop:
- runs forward pass,
- computes loss,
- backpropagates,
- clips gradients,
- updates parameters,
- saves checkpoints periodically.

The default step budget is modest so the notebook remains Colab-friendly.


In [ ]:
max_steps = 1500
save_every = 200

model.train()
step = start_step

for epoch in range(1000):
    for x, y in train_loader:
        if step >= max_steps:
            break

        x = x.to(device)
        y = y.to(device)

        # TODO: run the forward pass
        logits, loss = ...

        # TODO: zero gradients
        ...

        # TODO: backpropagate the loss
        ...

        # TODO: clip gradients
        ...

        # TODO: take an optimizer step
        ...

        # TODO: save the current step and loss for plotting
        ...
        ...

        if step % 50 == 0:
            print(f"step {step} | loss {loss.item():.4f}")

        if step > 0 and step % save_every == 0:
            save_checkpoint(model, opt, step, loss_steps, loss_history)

        step += 1

    if step >= max_steps:
        break


## 17. Validation

Compute the average validation loss.


In [ ]:
model.eval()
val_losses = []

print(f"Length of val loader: {len(val_loader)}")
with torch.no_grad():
    for x, y in tqdm(val_loader):
        x = x.to(device)
        y = y.to(device)

        # TODO: run the model on the validation batch
        logits, loss = ...

        # TODO: store the batch loss
        ...

val_loss = ...
print("val loss:", val_loss)


## 18. Autoregressive text generation

This section generates text from a prompt by repeatedly:
- conditioning on the current prefix,
- reading the logits at the final time step,
- sampling the next token,
- appending it to the sequence.


In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()

    for _ in range(max_new_tokens):
        # TODO: keep only the most recent context window
        idx_cond = ...

        # TODO: get logits from the model
        logits, _ = ...

        # TODO: take the logits from the final time step and apply temperature
        logits = ...

        if top_k is not None:
            # TODO: keep only the top-k logits
            v, _ = ...
            ...

        # TODO: convert logits to probabilities
        probs = ...

        # TODO: sample the next token
        next_token = ...

        # TODO: append the sampled token to the sequence
        idx = ...

    return idx

prompt = "Hi! How are you doing today?"

start_ids = ... # TODO: Encode prompt and move to device

out = generate(
    model,
    start_ids,
    max_new_tokens=50,
    temperature=1.0,
    top_k=2000,
)

print(tokenizer.decode(out[0].tolist()))

# Questions

In this section, analyze the behavior of your trained model. All the necessary data has been stored during training. Answer each question in the provided markdown or code cells, and ensure that all plots are clearly labeled.

## Q1: Plot Training Loss

### TODO:
- Use loss_steps and loss_history
- Plot loss vs steps
- Add labels, title, and grid

In [ ]:
# Q1: Plot Training Loss

import matplotlib.pyplot as plt

# TODO:
# - plot loss_history against loss_steps
# - label the axes
# - add a title, grid, and legend

...

### Answer 1:

# TODO

## Q2: What is the validation loss of your model? Is it lower or higher than the final training loss, and why?

### Answer 2:

# TODO

## Q3: What is the validation perplexity of your model? How does it relate to the validation loss, and what does it indicate about model performance?

In [ ]:
# Q3: Compute Validation Perplexity

import math

# TODO: compute validation perplexity from val_loss
val_perplexity = ...

print("validation perplexity:", val_perplexity)

### Answer 3:

# TODO

## Q4: How do temperature and top-k sampling affect determinism?

Run the `generate` function with different values of temperature and top-k, including extreme settings.

Based on your observations, answer:
- How does temperature affect determinism of the generated text?
- How does top-k affect determinism?
- Under what settings does generation become almost deterministic?
- Under what settings does it become highly random?


In [ ]:
# Q4: Experiment with temperature and top-k

### Answer 4:

# TODO